# 01 · Setup & Data (Colab)

在 Colab 跑这个 notebook：拉本地代码 → Kaggle 认证 → 下载数据 → 验证 import。

运行前：菜单 **Runtime → Change runtime type → T4 GPU**。

## Cell 1 · 拉本地代码 (从 GitHub)

In [19]:
# 先删旧 clone，保证每次重跑都拿到最新代码 + 正确分支（避免 Colab 残留旧版）
!rm -rf Kaggle-Study-
# -b 后跟你正在开发的分支（当前 process-data；以后合并到 main 后改成 main）
!git clone -b process-data https://github.com/SleepyEveryD/Kaggle-Study-.git
import sys
sys.path.append('/content/Kaggle-Study-/IAM-Handwritten-Forms-Dataset')

# 核对：应打印 process-data + 最新 commit（和本地一致）
!cd Kaggle-Study- && git branch --show-current && git log --oneline -1

# 注：改了 src 后重跑，最稳妥 Runtime → Restart session（避免 Python 模块缓存旧版）

Cloning into 'Kaggle-Study-'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (65/65), done.
Receiving objects: 100% (87/87), 32.33 KiB | 10.78 MiB/s, done.
remote: Total 87 (delta 37), reused 63 (delta 21), pack-reused 0 (from 0)
Resolving deltas: 100% (37/37), done.
process-data
6b79451 (HEAD -> process-data, origin/process-data) 持久化数据


## Cell 2 · Kaggle 认证

推荐用左侧 🔑 Secrets 建 `KAGGLE_USERNAME` / `KAGGLE_KEY`（并打开 Notebook access）。

In [20]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

# —— 或最简版（明文，仅自己用、别分享/别提交）：
# os.environ['KAGGLE_USERNAME'] = '你的用户名'
# os.environ['KAGGLE_KEY']      = '你复制的key'

!pip -q install kaggle

## Cell 3 · 下载数据（Google Drive 持久化）

策略：**只从 Kaggle 下载一次** → 把单个 zip 缓存到 Drive；**每次会话**从 Drive 解压到本地 `/content`（训练读本地才快，别直接读 Drive 上的海量小文件）。

首次运行会弹出 Drive 授权窗口，点同意即可。

In [21]:
import os, subprocess, shutil

# 挂载 Google Drive（首次弹授权窗口，点同意）
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/iam-handwritten-forms'   # Drive 持久化目录
ZIP_NAME  = 'iam-handwritten-forms-dataset.zip'
DRIVE_ZIP = os.path.join(DRIVE_DIR, ZIP_NAME)
LOCAL_DIR = '/content/data/iam'                              # 本次会话本地解压目录
os.makedirs(DRIVE_DIR, exist_ok=True)

# 1) Drive 上没有 zip → 从 Kaggle 下载一次，存进 Drive
if not os.path.exists(DRIVE_ZIP):
    print('⬇️ Drive 无缓存，从 Kaggle 下载...')
    subprocess.run('kaggle datasets download -d naderabdelghany/iam-handwritten-forms-dataset '
                   '-p /content/tmp', shell=True, check=True)
    shutil.move(f'/content/tmp/{ZIP_NAME}', DRIVE_ZIP)
    print('✅ 已缓存到 Drive：', DRIVE_ZIP)
else:
    print('✅ Drive 已有缓存，跳过 Kaggle 下载：', DRIVE_ZIP)

# 2) 本地还没解压 → 从 Drive 的 zip 解压到本地（每次会话；本地读训练才快）
if os.path.isdir(f'{LOCAL_DIR}/data') and os.listdir(f'{LOCAL_DIR}/data'):
    print('✅ 本地已解压，跳过')
else:
    print('📦 从 Drive 解压到本地 /content ...')
    os.makedirs(LOCAL_DIR, exist_ok=True)
    subprocess.run(f'unzip -q -o "{DRIVE_ZIP}" -d {LOCAL_DIR}', shell=True, check=True)
    print('✅ 解压完成')

!ls {LOCAL_DIR}

Mounted at /content/drive
⬇️ Drive 无缓存，从 Kaggle 下载...
✅ 已缓存到 Drive： /content/drive/MyDrive/iam-handwritten-forms/iam-handwritten-forms-dataset.zip
✅ 本地已解压，跳过
data  __notebook_source__.ipynb


## Cell 3.5 · 探查真实数据结构

真实数据在里层 `/content/data/iam/data/`。下面看清 forms/lines/words/ascii 的布局和标注文件位置。

In [22]:
# 1) 里层 data/ 下有哪些东西（顶层布局）
print('--- /content/data/iam/data ---')
!ls -la /content/data/iam/data

# 2) 全部 .txt 标注文件在哪、多大（words.txt / lines.txt / ascii 等）
print('\n--- 标注 txt 文件 ---')
!find /content/data/iam -maxdepth 3 -name "*.txt" -exec ls -lh {} \;

# 3) 各级目录树（限深度，避免刷屏）
print('\n--- 目录树 (maxdepth 3) ---')
!find /content/data/iam -maxdepth 3 -type d | head -40

# 4) 随便看一个标注文件的前 5 行（确认格式）—— 路径按上面结果调整
print('\n--- 某个 txt 头几行（示例，按实际路径改）---')
!find /content/data/iam -name "words.txt" -o -name "lines.txt" | head -1 | xargs head -n 8 2>/dev/null

--- /content/data/iam/data ---
total 2644
drwxr-xr-x 659 root root 12288 Jun  8 14:16 .
drwxr-xr-x   3 root root  4096 Jun  8 14:34 ..
drwxr-xr-x   2 root root  4096 Jun  8 14:16 000
drwxr-xr-x   2 root root  4096 Jun  8 14:16 001
drwxr-xr-x   2 root root  4096 Jun  8 14:16 002
drwxr-xr-x   2 root root  4096 Jun  8 14:16 003
drwxr-xr-x   2 root root  4096 Jun  8 14:16 004
drwxr-xr-x   2 root root  4096 Jun  8 14:16 005
drwxr-xr-x   2 root root  4096 Jun  8 14:16 006
drwxr-xr-x   2 root root  4096 Jun  8 14:16 007
drwxr-xr-x   2 root root  4096 Jun  8 14:16 008
drwxr-xr-x   2 root root  4096 Jun  8 14:16 009
drwxr-xr-x   2 root root  4096 Jun  8 14:16 010
drwxr-xr-x   2 root root  4096 Jun  8 14:16 011
drwxr-xr-x   2 root root  4096 Jun  8 14:16 012
drwxr-xr-x   2 root root  4096 Jun  8 14:16 013
drwxr-xr-x   2 root root  4096 Jun  8 14:16 014
drwxr-xr-x   2 root root  4096 Jun  8 14:16 015
drwxr-xr-x   2 root root  4096 Jun  8 14:16 016
drwxr-xr-x   2 root root  4096 Jun  8 14:16 017
d

## Cell 4 · 验证能 import 本地代码 + 路径就绪

In [23]:
from src import config
print('ENV =', config.ENV)
print('DATA_DIR =', config.DATA_DIR)

ENV = colab
DATA_DIR = /content/data/iam
